# 1. Introduction
*   **Weights & Biases Link:** TODO [Insert the link to your W&B report/view here] [1].
*   **Tools & Sources Used:**
    - NotebookLM: Summarizing course materials and project description, Creating Template of Python Notebook
    - Gemini: Coding support / Learning help
    - Claude: Coding support / Coding Coach


# 2. Setup
*State and justify your setup decisions here [1].*

In [ ]:
# !pip install datasets gensim wandb nltk fasttext torch

In [1]:
# Import all necessary libraries (e.g., PyTorch, Lightning, Hugging Face Datasets, Weights & Biases) [2, 4, 5]
import torch
from datasets import load_dataset
import wandb
import nltk
import numpy as np
import random

/Users/janis.kneubuehler/workspace/NLP/NLP_Exercises/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
# Reproducibility with seed 42
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 3. Preprocessing & Data Loading
**Decisions:**
- Tokenization: I use nltk.word_tokenize as recommended in the project discussion and demonstrated in the exercise of SW02.
- Lowercase: I convert to lowercase to reduce the vocabulary size, therefore we have less unknown words
- Removal of punctuation: I remove punctuation for reducing vocabulary size and because they have no semantic meaning. with the nltk.word_tokenize they are separate tokens and easy to remove. 
- Removal of stopwords: I don't remove stopwerds, because default stopwords like "in", "on", etc. are critical for understanding physical relationships. With removal of them the project wouldn't make any sense.
- Stemming and Lemmatization: I skip stemming and lemmatizing because the word2vec is huge and has different word forms in it. Stemming would maybe perform worse, because some "not-real words" wouldn't be recognized by word2vec.
- Cleaning unknown words: I map unkknown words into an <unk> token. This was suggested by Gemini as it is a simple way to prevent KeyErros while training.  
- Format cleaning: No html removal necessary, because it is already cleaned in the dataset. (see Data Exploration)
- Truncation: No truncation made due to TODO
- Feature selection: TODO
- Input format: I concatenate the goal with both solution and a separator, so from one row i get two: goal + <SEP> + solution1 and goal + <SEP> + solution2. TODO
- Label format: A boolean if solution1 or solution2 is correct.
- Batching, padding: TODO
- Vocabulary, Embedding: TODO

In [26]:
# Load the PIQA dataset from revision because dataset scripts are no longer supported
# Use splits like specified in Course Project Slides
train = load_dataset('ybisk/piqa', split='train[:-1000]', revision='refs/convert/parquet')
valid = load_dataset('ybisk/piqa', split='train[-1000:]', revision='refs/convert/parquet')
test = load_dataset('ybisk/piqa', split='validation', revision='refs/convert/parquet')

In [27]:
import re

# Exploring the data
print(f"training: {len(train)}, validation: {len(valid)}, test: {len(test)}")

print("Exploring some random rows in dataset:")
print(train[0])
print(train[12])
print(train[123])
print(train[4563])
print(train[13245]) 

# Count rows with html (Regex was generated with Gemini and validated by me with https://regex101.com/)
goals_with_html = train.filter(lambda x: re.search(r'</?[^>]+>', x['goal'], re.IGNORECASE))
sols1_with_html = train.filter(lambda x: re.search(r'</?[^>]+>', x['sol1'], re.IGNORECASE))
sols2_with_html = train.filter(lambda x: re.search(r'</?[^>]+>', x['sol2'], re.IGNORECASE))

print(f"Number of rows with html (for format cleaning): {len(goals_with_html)}, {len(sols1_with_html)}, {len(sols2_with_html)}")

print(f"Class distribution in train: 0: {sum(1 for ex in train if ex['label']==0)} and 1: {sum(1 for ex in train if ex['label']==1)}")

training: 15113, validation: 1000, test: 1838
Exploring some random rows in dataset:
{'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1}
{'goal': 'how do you open a capri-sun', 'sol1': 'open the straw attached to the juice, and then stick it in the small hole at the front of the pouch.', 'sol2': 'cut the top off with scissors', 'label': 0}
{'goal': 'How do you open a gift without ripping the paper?', 'sol1': 'Hold gift over boiling water to let the steam release the tape adhesive.', 'sol2': 'Hold gift over cold water to let the steam release the tape adhesive.', 'label': 0}
{'goal': 'how to make fancy piped flowers', 'sol1': 'use molded ganache', 'sol2': 'use Russian piping tips', 'label': 1}
{'goal': 'How do you keep steel wool and cotton dry?', 'sol1': 'Place the make-shift fire starter inside a film canister.', 'sol2': 'Place the make-shift fire starter inside your pocket.', 'label': 0}
Number of rows wi

In [28]:
# Workaround on mac for SSL certificate error (Code by Gemini) TODO delete
import os
import ssl
import certifi

# 1. Point the notebook to your trusted certificates
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# 2. Force Python to allow unverified contexts globally (The safety net)
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

print("SSL Bypass active!")

SSL Bypass active!


In [29]:
import string

def preprocess(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [token for token in tokens if token not in string.punctuation]

    return tokens

def preprocess_data(data):
    goal_tokens = preprocess(data['goal'])
    sol1_tokens = preprocess(data['sol1'])
    sol2_tokens = preprocess(data['sol2'])
    
    # concat goal with input
    data['input1_tokens'] = goal_tokens + ['<sep>'] + sol1_tokens
    data['input2_tokens'] = goal_tokens + ['<sep>'] + sol2_tokens
    
    return data

train = train.map(preprocess_data)
valid = valid.map(preprocess_data)
test = test.map(preprocess_data)

print(train[0])


{'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1, 'input1_tokens': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can', '<sep>', 'pour', 'it', 'onto', 'a', 'plate'], 'input2_tokens': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can', '<sep>', 'pour', 'it', 'into', 'a', 'jar']}


**Decisions specific for Vocabulary:**
- I only use training data for the vocabulary to prevent data leakage into validation or test
- It would be possible to do the vocabulary with torchtext. I didn't use torchtext because it was not part of our course material.
- I added special tokens for padding (pad), unknown words (unk) and separator for separating goal and solution (sep).
- I did not analyse the unknown words of test set to get not biased.
This class was built with help of Claude.

In [30]:
class Vocabulary:
    def __init__(self):
        self.word2idx = {'<pad>': 0, '<unk>': 1, '<sep>': 2}
        self.idx2word = {0: '<pad>', 1: '<unk>', 2: '<sep>'}
        self.idx = 3

    def build_vocab(self, dataset):
        for example in dataset:
            for token in example['input1_tokens'] + example['input2_tokens']:
                if token not in self.word2idx:
                    self.word2idx[token] = self.idx
                    self.idx2word[self.idx] = token
                    self.idx += 1

    def encode(self, tokens):
        encoded = []
        unk_count = 0
        for token in tokens:
            if token in self.word2idx:
                encoded.append(self.word2idx[token])
            else:
                encoded.append(self.word2idx['<unk>'])
                unk_count += 1
        return encoded, unk_count

vocab = Vocabulary()
vocab.build_vocab(train)
print(f"Total unique words in Training Vocabulary: {vocab.idx}")

def encode_data(example):
    enc1, unk1 = vocab.encode(example['input1_tokens'])
    enc2, unk2 = vocab.encode(example['input2_tokens'])
    
    example['input1_ids'] = enc1
    example['input2_ids'] = enc2
    
    example['unk_count'] = unk1 + unk2
    return example

train = train.map(encode_data)
valid = valid.map(encode_data)

train_unks = sum(train['unk_count'])
valid_unks = sum(valid['unk_count'])
train_total = sum(len(ex['input1_ids']) + len(ex['input2_ids']) for ex in train)
valid_total  = sum(len(ex['input1_ids']) + len(ex['input2_ids']) for ex in valid)
print(f"<unk> tokens – Train: {train_unks} ({100*train_unks/train_total:.2f}%)")
print(f"<unk> tokens – Valid:  {valid_unks} ({100*valid_unks/valid_total:.2f}%)")


Total unique words in Training Vocabulary: 15766
<unk> tokens – Train: 0 (0.00%)
<unk> tokens – Valid:  781 (1.41%)


**Decisions specific for DataLoader:**
- With pad_sequence I pad each batch to the longest sequence (dynamic padding). This was suggested by Gemini due to efficeny.
- I could easily train my Model 1 with a bigger batch size (up to 1024), but then i would only have 15 learning steps per epoch. This is why I use only batch size of 32. (=468 steps) <br>I fixed this hyperparameter at first.
- I shuffle train data to avoid the model to memorize the order in the dataset.

In [31]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    input1_list = []
    input2_list = []
    labels = []
    
    for example in batch:
        input1_list.append(torch.tensor(example['input1_ids']))
        input2_list.append(torch.tensor(example['input2_ids']))
        labels.append(example['label'])
        
    input1_padded = pad_sequence(input1_list, batch_first=True, padding_value=0)
    input2_padded = pad_sequence(input2_list, batch_first=True, padding_value=0)
    
    labels_tensor = torch.tensor(labels, dtype=torch.long)
    
    return {
        'input1': input1_padded, 
        'input2': input2_padded, 
        'labels': labels_tensor
    }

batch_size = 32 

train_loader = DataLoader(train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

first_batch = next(iter(train_loader))
print("Input 1 Batch Shape:", first_batch['input1'].shape)
print("Labels Batch Shape:", first_batch['labels'].shape)

Input 1 Batch Shape: torch.Size([32, 90])
Labels Batch Shape: torch.Size([32])


# 4. Model Definition
*State and justify your architectural decisions here [1].*
TODO why word2vec

**Architectures to Implement [7, 8]:**
1.  **Architecture 1:** Pre-trained Word Embeddings (choose word2vec, GloVe, or fastText) + a 2-layer classifier with a ReLU non-linearity. Train *only* the classifier.
2.  **Architecture 2:** A 2-layer RNN (use PyTorch's LSTM or GRU) + a 2-layer classifier with a ReLU. Use the exact same word embeddings from Architecture 1 for the input layer, but train the whole network end-to-end.

In [32]:
import gensim.downloader as api

# Load word2vec embeddings, copied from here: https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html 
wv = api.load('word2vec-google-news-300')
print(f"Vector size: {wv.vector_size}, Vocabulary size: {len(wv.index_to_key)}")

# Analyse word2vec
for index, word in enumerate(wv.index_to_key):
    if index == 10:
        break
    print(f"word #{index}/{len(wv.index_to_key)} is {word}")

Vector size: 300, Vocabulary size: 3000000
word #0/3000000 is </s>
word #1/3000000 is in
word #2/3000000 is for
word #3/3000000 is that
word #4/3000000 is is
word #5/3000000 is on
word #6/3000000 is ##
word #7/3000000 is The
word #8/3000000 is with
word #9/3000000 is said


In [33]:
import torch.nn as nn

embedding_dim = wv.vector_size
vocab_size = len(vocab.word2idx)

# Gemini helped here
# embedding_matrix = torch.randn((vocab_size, embedding_dim)) * 0.6
# embedding_matrix[vocab.word2idx['<pad>']] = torch.zeros(embedding_dim)

# Claude helped here
embedding_matrix = torch.empty(vocab_size, embedding_dim)
nn.init.uniform_(embedding_matrix, -1.0 / embedding_dim, 1.0 / embedding_dim)
embedding_matrix[vocab.word2idx['<pad>']] = torch.zeros(embedding_dim)

words_found = 0
for word, idx in vocab.word2idx.items():
    if word in wv:
        embedding_matrix[idx] = torch.from_numpy(np.array(wv[word], dtype=np.float32))
        words_found += 1

print(f"Filled {words_found} out of {vocab_size} words with pre-trained Word2Vec vectors. ({100*words_found/vocab_size:.1f}%) \nThe rest ist filled with random init.")

Filled 13348 out of 15766 words with pre-trained Word2Vec vectors. (84.7%) 
The rest ist filled with random init.


In [34]:
class ArchitectureOne(nn.Module):
    
    def __init__(self, embedding_matrix, hidden_dim=128, dropout_rate=0.1):
        super().__init__()
        
        # Frozen pre-trained embeddings; padding_idx=0 zeroes out <pad> gradients (corrected by Claude)
        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix, freeze=True, padding_idx=0
        )
        
        embedding_dim = embedding_matrix.shape[1]
        
        # refactored by Claude
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim, 1),
        )

    def _mean_pool(self, input_ids):
        embeds = self.embedding(input_ids)
        mask   = (input_ids != 0).float().unsqueeze(-1) # exclude padding
        summed = (embeds * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1e-9)
        return summed / lengths

    def forward(self, input1, input2):
        # Score each option independently
        score1 = self.classifier(self._mean_pool(input1))
        score2 = self.classifier(self._mean_pool(input2))

        logits = torch.cat([score1, score2], dim=1)
        return logits

In [ ]:
# Define Architecture 2 (RNN + Classifier)
class ArchitectureTwo(torch.nn.Module):
    def __init__(self, embedding_matrix, lstm_hidden_dim=128, classifier_hidden_dim=128, num_layers=2, dropout_rate=0.1):
        super().__init__()
        
        embedding_dim = embedding_matrix.shape[1]
        
        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix, freeze=False, padding_idx=0
        )
        
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout_rate
        )
        
        lstm_output_dim = 2 * lstm_hidden_dim
        
        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_dim, classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(classifier_hidden_dim, 1)
        )
        
    def _encode_sequence(self, input_ids):
        embeds = self.embedding(input_ids)
        _, (hn, _) = self.lstm(embeds)
        
        sentence_vec = torch.cat([hn[-2], hn[-1]], dim=-1)
        return sentence_vec
    
    def forward(self, input1, input2):
        score1 = self.classifier(self._encode_sequence(input1))
        score2 = self.classifier(self._encode_sequence(input2))

        logits = torch.cat([score1, score2], dim=1)
        return logits

ArchitectureTwo(
  (embedding): Embedding(15766, 300, padding_idx=0)
  (lstm): LSTM(300, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=1, bias=True)
  )
)


# 5. Training
*State and justify your training decisions here [1].*

*   Discuss your chosen loss function and optimization algorithm.
*   Detail the hyperparameters you are using.


In [ ]:
def train_model(model, train_loader, valid_loader, criterion, optimizer, checkpoint_name, epochs=20, patience=3):
    best_val_accuracy = 0.0
    patience_counter = 0
    checkpoint_best_path = f"best_{checkpoint_name}_model.pt"
    checkpoint_last_path = f"last_{checkpoint_name}_model.pt" # Save last model if my run gets aborted
    print(f"Number of batches {len(train_loader)}")

    for epoch in range(epochs):
        ########################## Training ##########################
        model.train()
        total_train_loss = 0.0
        
        for batch in train_loader:
            optimizer.zero_grad()
            logits = model(batch['input1'], batch['input2'])
            loss   = criterion(logits, batch['labels'])
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            
        avg_train_loss = total_train_loss / len(train_loader)
        
        ########################## Validation ##########################
        model.eval()
        total_val_loss = 0.0
        correct_preds = 0
        total_preds = 0
        
        with torch.no_grad(): # TODO No Grand begründen
            for batch in valid_loader:
                logits = model(batch['input1'], batch['input2'])
                loss   = criterion(logits, batch['labels'])
                total_val_loss += loss.item()
                
                # Calculate accuracy
                predictions = torch.argmax(logits, dim=1)
                correct_preds += (predictions == batch['labels']).sum().item()
                total_preds += batch['labels'].size(0)
                
        avg_val_loss = total_val_loss / len(valid_loader)
        val_accuracy = correct_preds / total_preds
        
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_accuracy": val_accuracy
        })
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")
        
        ########################## Checkpoints ##########################
        torch.save(model.state_dict(), checkpoint_last_path)
        
        if val_accuracy > best_val_accuracy: # Best model & early stopping based on validation accuracy (as discussed in project discussion)
            print(f"Validation accuracy improved from {best_val_accuracy:.4f} to {val_accuracy:.4f}. Saving checkpoint.")
            best_val_accuracy = val_accuracy
            patience_counter = 0
            
            torch.save(model.state_dict(), checkpoint_best_path)
        else:
            patience_counter += 1
            print(f"No improvement in validation accuracy. Patience: {patience_counter}/{patience}")
            
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs!")
                break # Exit the training loop

    print(f"Training complete. Saved best and last model.")

In [ ]:
import torch.optim as optim

# Login to W&B
wandb.login()

wandb.init(
    project="janis-kneubuehler-nlp-project1", 
    config={
        "architecture": "ArchitectureOne",
        "run_goal": "decrease dropout",
        "learning_rate": 1e-3,
        "hidden_dim": 128,
        "dropout": 0.2,
        "epochs": 30,
        "batch_size": batch_size
    }
)
config = wandb.config

model = ArchitectureOne(embedding_matrix, hidden_dim=128, dropout_rate=config.dropout)

# I use CrossEntropyLoss and not BCELoss. It is easier to understand for me and does not need a Sigmoid function. It should not make a difference in quality.
criterion = nn.CrossEntropyLoss()
# I use Adam optimizer as discussed in the project discussion
trainable_parameters = (p for p in model.parameters() if p.requires_grad)
optimizer = optim.Adam(trainable_parameters, lr=config.learning_rate)

train_model(model, train_loader, valid_loader, criterion, optimizer, checkpoint_name="arch1", epochs=config.epochs, patience=6)

wandb.finish()


Number of batches 473
Epoch 1/30 | Train Loss: 0.6923 | Val Loss: 0.6913 | Val Acc: 0.5250
Validation accuracy improved from 0.0000 to 0.5250. Saving checkpoint.
Epoch 2/30 | Train Loss: 0.6900 | Val Loss: 0.6892 | Val Acc: 0.5610
Validation accuracy improved from 0.5250 to 0.5610. Saving checkpoint.
Epoch 3/30 | Train Loss: 0.6864 | Val Loss: 0.6881 | Val Acc: 0.5500
No improvement in validation accuracy. Patience: 1/6
Epoch 4/30 | Train Loss: 0.6836 | Val Loss: 0.6870 | Val Acc: 0.5560
No improvement in validation accuracy. Patience: 2/6
Epoch 5/30 | Train Loss: 0.6802 | Val Loss: 0.6865 | Val Acc: 0.5560
No improvement in validation accuracy. Patience: 3/6
Epoch 6/30 | Train Loss: 0.6794 | Val Loss: 0.6855 | Val Acc: 0.5610
No improvement in validation accuracy. Patience: 4/6
Epoch 7/30 | Train Loss: 0.6756 | Val Loss: 0.6852 | Val Acc: 0.5590
No improvement in validation accuracy. Patience: 5/6
Epoch 8/30 | Train Loss: 0.6722 | Val Loss: 0.6847 | Val Acc: 0.5640
Validation accuracy

epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
train_loss,██▇▇▇▇▇▆▆▆▅▅▅▄▄▄▃▃▃▂▂▂▁
val_accuracy,▁▆▄▅▅▆▅▆▇▆▇▆▆▆▇▇█▇█▆▇▆█
val_loss,█▆▆▅▄▄▄▃▂▂▁▂▂▂▁▂▁▂▁▂▄▅▃
epoch,23
train_loss,0.61435
val_accuracy,0.577
val_loss,0.68395


In [ ]:
import torch.optim as optim

# Login to W&B
wandb.login()

wandb.init(
    project="janis-kneubuehler-nlp-project1", 
    config={
        "architecture": "ArchitectureTwo",
        "run_goal": "initial",
        "learning_rate": 1e-3,
        'lstm_hidden_dim': 128,
        'classifier_hidden_dim': 128,
        'num_layers': 2,
        "dropout": 0.1,
        "epochs": 30,
        "batch_size": batch_size
    }
)
config2 = wandb.config

model2 = ArchitectureTwo(embedding_matrix, lstm_hidden_dim=config2.lstm_hidden_dim, classifier_hidden_dim=config2.classifier_hidden_dim, num_layers=config2.num_layers, dropout_rate=config2.dropout)
criterion2  = nn.CrossEntropyLoss()
optimizer2  = optim.Adam(model2.parameters(), lr=config2.learning_rate)

train_model(model2, train_loader, valid_loader, criterion2, optimizer2, checkpoint_name="arch2", epochs=config2.epochs, patience=6)

wandb.finish()




# 6. Evaluation

In [38]:
# Add your code to evaluate both architectures on the validation set
# Add your code for the final evaluation on the test set

# 7. Conclusion & Interpretation
*State and justify your conclusions here [1].*

*   Interpret your results. Discuss the specialties of each model type, how they handled the physical commonsense reasoning task, and summarize your findings [1, 9].